In [ ]:
import sys
import os
current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, '..'))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)



In [ ]:
import torch
import random
import numpy as np
import csv

In [ ]:
from models.semcovert import create_network
from utils.video_utils import load_video_as_tensor,split_video_tensor, get_video_files
from utils.metrics import calculate_psnr, calculate_ssim, calculate_fvd

In [ ]:
def seet_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seet_seed(42)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator, AutoMinorLocator
from matplotlib.ticker import FormatStrFormatter

def plot_custom_line_chart(data_groups, 
                           x_labels, 
                           y_label, 
                           legend_labels,
                           title="Custom Line Chart",
                           str_format='%.2f',
                           colors=None,
                           markers=None,
                           linestyles=None,
                           save_path=None):
    
    plt.figure(figsize=(16, 10))
    
    default_colors = ['#0072B2', '#E69F00', '#882255', '#009E73',
                               '#CC79A7', '#D55E00', '#56B4E9', '#F0E442',
                               '#999999', '#FF69B4', '#8B4513', '#2F4F4F',]

    default_markers = ['o', 's', '^', 'D', 'v', '<', '>', 'p', '*', 'h']
    default_linestyles = ['-', '--', '-.', ':', (0, (3, 1, 1, 1)), (0, (5, 1)), (0, (3, 5, 1, 5))]
    
    legend_handles = []  
    
    for i, (x_data, y_data) in enumerate(data_groups):
        color = colors[i] if colors and i < len(colors) else default_colors[i]
        marker = markers[i] if markers and i < len(markers) else default_markers[i % len(default_markers)]
        linestyle = linestyles[i] if linestyles and i < len(linestyles) else default_linestyles[i % len(default_linestyles)]

        plt.plot(x_data, y_data, 
                 linestyle=linestyle, 
                 linewidth=3.0,
                 color=color,
                 alpha=0.85,
                 zorder=1)
        
        plt.plot(x_data, y_data, 
                 marker=marker,
                 linestyle='none',  
                 markersize=14,
                 markeredgewidth=3.0,
                 markeredgecolor=color,
                 markerfacecolor='white',  
                 color=color,
                 zorder=10)

        custom_handle = Line2D([0], [0],
                               linestyle=linestyle,
                               linewidth=2.5,
                               color=color,
                               marker=marker,
                               markersize=10,
                               markeredgewidth=2.5,
                               markeredgecolor=color,
                               markerfacecolor='white')
        legend_handles.append(custom_handle)
    
    plt.xlabel(x_labels[0], fontsize=32, fontweight='normal')
    plt.ylabel(y_label, fontsize=32, fontweight='normal')
    
    # plt.title(title, fontsize=16, fontname='Times New Roman')

    plt.legend(handles=legend_handles, labels=legend_labels, loc='best', fontsize=30, framealpha=0.7, edgecolor='none')
    
    plt.xticks(fontsize=32)
    plt.yticks(fontsize=32)
    
    ax = plt.gca()
    ax.grid(True, which='major', linestyle='-', linewidth=1.2, alpha=0.6)
    ax.grid(True, which='minor', linestyle=':', linewidth=0.8, alpha=0.4)
    
    ax.xaxis.set_major_locator(MultipleLocator(2))  
    ax.minorticks_on()
    ax.xaxis.set_minor_locator(AutoMinorLocator(2))  
    ax.yaxis.set_minor_locator(AutoMinorLocator(2))

    ax.yaxis.set_major_formatter(FormatStrFormatter(str_format))  # Format y-axis labels
    
    ax.set_facecolor("#ffffff")
    for spine in ax.spines.values():
        spine.set_linewidth(2)  
        spine.set_color("#000000")
    
    plt.tight_layout()
    
    if save_path:
        if not os.path.exists(os.path.dirname(save_path)):
            os.makedirs(os.path.dirname(save_path))
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Fig Saved: {save_path}")
    
    plt.show()

In [ ]:
def load_videos(data_dir, target_size=(240,320), max_frames=5): 
    
    video_files = get_video_files(data_dir)
    if not video_files:
        raise ValueError(f"No video files found in {data_dir}")
    
    print(f"Found {len(video_files)} video files: {[f.name for f in video_files]}")
    
    all_video_batches = []
    for video_file in video_files:
        video_tensor = load_video_as_tensor(
            str(video_file), 
            target_size=target_size
        )
        video_tensor = split_video_tensor(video_tensor, max_frames)
        if video_tensor.numel() > 0:
            for i in range(video_tensor.shape[0]):
                all_video_batches.append(video_tensor[i])
            del video_tensor  
        else:
            print(f"  Failed to load video: {video_file}")
    
    if not all_video_batches:
        raise ValueError("No valid video data loaded")
    
    all_video_tensor = torch.stack(all_video_batches, dim=0)
    del all_video_batches

    indices = torch.randperm(all_video_tensor.shape[0])
    if len(indices) % 2 != 0:
        indices = indices[:-1] # Ensure even number of videos
        
    mid_count = len(indices) // 2
    cover_indices = indices[:mid_count]
    secret_indices = indices[mid_count:]

    cover_videos = all_video_tensor[cover_indices]
    secret_videos = all_video_tensor[secret_indices]
    del all_video_tensor  
    print(f"Loaded {cover_videos.shape[0]} cover videos and secret videos")
    
    return cover_videos, secret_videos

In [ ]:
def generate(model, cover_video, secret_video, batch_size, device):
    recon_cover = torch.empty_like(cover_video)
    if secret_video is not None:
        recon_secret = torch.empty_like(secret_video) 
    else:
        recon_secret = None
    with torch.no_grad():
        for i in range(0, cover_video.shape[0], batch_size):
            batch_cover = cover_video[i:i + batch_size].to(device)
            if secret_video is not None:
                batch_secret = secret_video[i:i + batch_size].to(device) 
            else:
                batch_secret = None
   
            output = model(batch_cover, batch_secret)
            recon_cover_chunk = output['cover_reconstructed'].cpu()
            recon_cover[i:i + batch_size] = recon_cover_chunk
            if secret_video is not None:
                recon_secret_chunk = output['secret_reconstructed'].cpu()
                recon_secret[i:i + batch_size] = recon_secret_chunk

    return recon_cover, recon_secret

In [ ]:
import torchvision.models.video as model_3d
from torchvision.models.video import S3D_Weights
import torch.nn as nn
s3d = nn.Sequential(
    model_3d.s3d(weights=S3D_Weights.DEFAULT).features,
    nn.AdaptiveAvgPool3d((1, 1, 1))
)

s3d.eval()

In [ ]:
model_cfg = {
        'depth': 4,
        'dim': 96,
        'use_channel': True,
        'vae_config': {
            'dim': 96,
            'z_dim': 16,
            'dim_mult': [1, 2, 4, 4],
            'num_res_blocks': 2,
            'attn_scales': [],
            'temperal_downsample': [False, True, True],
            'dropout': 0.0
        },
        'channel_config': {
            'channel_type': 'AWGN',  # Channel type
            'snr': 25,               # Signal-to-noise ratio
        },    
    }

In [ ]:
def run(model, video_dir, channel_config, saved_path, target_size=(240,320), max_frames=5, max_batch=1000, batch_size=8, device='cuda'):
    cover_videos, secret_videos = load_videos(video_dir, target_size=target_size, max_frames=max_frames)
    
    # Limit the number of videos to max_batch to avoid memory issues
    cover_videos = cover_videos[:max_batch]
    secret_videos = secret_videos[:max_batch]
    print("Short the videos to max_batch:", max_batch)

    model.to(device)
    model.eval()
    cover_psnr_list = []
    secret_psnr_list = []
    cover_ssim_list = []
    secret_ssim_list = []
    cover_fvd_list = []
    secret_fvd_list = []
    diff_snr = channel_config['snr']
    channel_type = channel_config['channel_type']

    for snr in diff_snr:
        print(f"Testing with SNR: {snr} dB, channel type: {channel_type}")
        channel_config = {
            'channel_type': channel_type,  # Channel type
            'snr': snr,               # Signal-to-noise ratio
        }
        model.set_channel(channel_config)
        
        recon_cover, recon_secret = generate(model, cover_videos, secret_videos, batch_size, device)
        
        cover_psnr = calculate_psnr(recon_cover, cover_videos)
        secret_psnr = calculate_psnr(recon_secret, secret_videos)
        cover_ssim = calculate_ssim(recon_cover, cover_videos)
        secret_ssim = calculate_ssim(recon_secret, secret_videos)
        cover_fvd = calculate_fvd(model_3d=s3d, real_videos=cover_videos, fake_videos=recon_cover, 
                            device=device, batch_size=batch_size)
        secret_fvd = calculate_fvd(model_3d=s3d, real_videos=secret_videos, fake_videos=recon_secret, 
                            device=device, batch_size=batch_size)
        
        print(f"Cover PSNR: {cover_psnr:.2f} dB, Secret PSNR: {secret_psnr:.2f} dB")
        print(f"Cover SSIM: {cover_ssim:.4f}, Secret SSIM: {secret_ssim:.4f}")
        print(f"Cover FVD: {cover_fvd:.2f}, Secret FVD: {secret_fvd:.2f}")
        
        cover_psnr_list.append(cover_psnr)
        secret_psnr_list.append(secret_psnr)
        cover_ssim_list.append(cover_ssim)
        secret_ssim_list.append(secret_ssim)
        cover_fvd_list.append(cover_fvd)
        secret_fvd_list.append(secret_fvd)

        del recon_cover, recon_secret  # Free memory

    only_cover_psnr_list = []
    only_cover_ssim_list = []
    only_cover_fvd_list = []
    # Test for only cover videos
    for snr in diff_snr:
        print(f"Testing with SNR: {snr} dB, channel type: {channel_type}")
        channel_config = {
            'channel_type': channel_type,  # Channel type
            'snr': snr,               # Signal-to-noise ratio
        }
        model.set_channel(channel_config)
        
        recon_cover,_ = generate(model, cover_videos, None, batch_size, device)
        
        cover_psnr = calculate_psnr(recon_cover, cover_videos)
        cover_ssim = calculate_ssim(recon_cover, cover_videos)
        cover_fvd = calculate_fvd(model_3d=s3d, real_videos=cover_videos, fake_videos=recon_cover, 
                            device=device, batch_size=batch_size)
        
        print(f"Cover PSNR: {cover_psnr:.2f} dB")
        print(f"Cover SSIM: {cover_ssim:.4f}")
        print(f"Cover FVD: {cover_fvd:.2f}")

        only_cover_psnr_list.append(cover_psnr)
        only_cover_ssim_list.append(cover_ssim)
        only_cover_fvd_list.append(cover_fvd)

        del recon_cover  # Free memory

    only_sercert_psnr_list = []
    only_sercert_ssim_list = []
    only_sercert_fvd_list = []
    # Test for only secret videos
    for snr in diff_snr:
        print(f"Testing with SNR: {snr} dB, channel type: {channel_type} ")
        channel_config = {
            'channel_type': channel_type,  # Channel type
            'snr': snr,               # Signal-to-noise ratio
        }
        model.set_channel(channel_config)
        
        recon_secret, _ = generate(model, secret_videos, None, batch_size, device)
        
        secret_psnr = calculate_psnr(recon_secret, secret_videos)
        secret_ssim = calculate_ssim(recon_secret, secret_videos)
        secret_fvd = calculate_fvd(model_3d=s3d, real_videos=secret_videos, fake_videos=recon_secret, 
                            device=device, batch_size=batch_size)
        
        print(f"Secret PSNR: {secret_psnr:.2f} dB")
        print(f"Secret SSIM: {secret_ssim:.4f}")
        print(f"Secret FVD: {secret_fvd:.2f}")

        only_sercert_psnr_list.append(secret_psnr)
        only_sercert_ssim_list.append(secret_ssim)
        only_sercert_fvd_list.append(secret_fvd)

        del recon_secret  # Free memory


    plot_custom_line_chart(
            data_groups=[
                (diff_snr, only_cover_psnr_list),
                (diff_snr, only_sercert_psnr_list),
                (diff_snr, cover_psnr_list),
                (diff_snr, secret_psnr_list)
            ],
            x_labels=['SNR (dB)'],
            y_label='PSNR (dB)',
            legend_labels=['Cover Videos (SemCom)', 'Secret Videos (SemCom)', 'Cover Videos (SemCovert)', 'Secret Videos (SemCovert)'],
            str_format='%.0f',
            save_path= str(saved_path)+'/psnr_comparison.pdf'
        )
    plot_custom_line_chart(
            data_groups=[
                (diff_snr, only_cover_ssim_list),
                (diff_snr, only_sercert_ssim_list),
                (diff_snr, cover_ssim_list),
                (diff_snr, secret_ssim_list)
            ],
            x_labels=['SNR (dB)'],
            y_label='SSIM',
            legend_labels=['Cover Videos (SemCom)', 'Secret Videos (SemCom)', 'Cover Videos (SemCovert)', 'Secret Videos (SemCovert)'],
            str_format='%.2f',
            save_path=str(saved_path)+'/ssim_comparison.pdf'
        )
    plot_custom_line_chart(
            data_groups=[
                (diff_snr, only_cover_fvd_list),
                (diff_snr, only_sercert_fvd_list),
                (diff_snr, cover_fvd_list),
                (diff_snr, secret_fvd_list)
            ],
            x_labels=['SNR (dB)'],
            y_label='FVD',
            legend_labels=['Cover Videos (SemCom)', 'Secret Videos (SemCom)', 'Cover Videos (SemCovert)', 'Secret Videos (SemCovert)'],
            str_format='%.1f',
            save_path=(saved_path)+'/fvd_comparison.pdf'
        )

    metrics_saved_file = str(saved_path)+'/metrics.csv'
    
    with open(metrics_saved_file, 'w', newline='') as csvfile:
        title = ['SNR (dB)', 'Cover PSNR', 'Secret PSNR', 'Only Cover PSNR', 'Only Secret PSNR',
                'Cover SSIM', 'Secret SSIM', 'Only Cover SSIM', 'Only Secret SSIM',
                'Cover FVD', 'Secret FVD', 'Only Cover FVD', 'Only Secret FVD']
        writer = csv.writer(csvfile)
        writer.writerow(title)
        for i in range(len(diff_snr)):
            row = [
                diff_snr[i],
                cover_psnr_list[i],
                secret_psnr_list[i],
                only_cover_psnr_list[i],
                only_sercert_psnr_list[i],
                cover_ssim_list[i],
                secret_ssim_list[i],
                only_cover_ssim_list[i],
                only_sercert_ssim_list[i],
                cover_fvd_list[i],
                secret_fvd_list[i],
                only_cover_fvd_list[i],
                only_sercert_fvd_list[i]
            ]
            writer.writerow(row)

In [ ]:
video_dir = "YOUR_VIDEO_DIR"  # Path to your video directory
# Test for UCF101 Dataset
diff_snr = [5,9,13,17,21,25,29,33]  # Different SNR values
channel_type = 'AWGN'  # Channel type

model_path = "YOUR_MODEL_PATH"  # Path to your model weights
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = create_network(model_cfg)
model.load_state_dict(torch.load(model_path, map_location=device))

saved_path = "YOUR_SAVE_PATH"  # Path to save the results
run(model, video_dir, 
    channel_config={'channel_type': channel_type, 'snr': diff_snr}, 
    saved_path=saved_path, 
    target_size=(240,320), 
    max_frames=5, 
    max_batch=1000,
    batch_size=16, 
    device=device)


In [ ]:
video_dir = "YOUR_VIDEO_DIR"  # Path to your video directory
# Test for DAVIS Dataset
diff_snr = [5,9,13,17,21,25,29,33]  # Different SNR values
channel_type = 'AWGN'  # Channel type

model_path = "YOUR_MODEL_PATH"  # Path to your model weights
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = create_network(model_cfg)
model.load_state_dict(torch.load(model_path, map_location=device))

saved_path = "YOUR_SAVE_PATH"  # Path to save the results
run(model, video_dir, 
    channel_config={'channel_type': channel_type, 'snr': diff_snr}, 
    saved_path=saved_path, 
    target_size=(480,640), 
    max_frames=5, 
    max_batch=500,
    batch_size=4, 
    device=device)

In [ ]:
video_dir = "YOUR_VIDEO_DIR"  # Path to your video directory
# Test for MOT17 Dataset
diff_snr = [5,9,13,17,21,25,29,33]  # Different SNR values
channel_type = 'AWGN'  # Channel type

model_path = "YOUR_MODEL_PATH"  # Path to your model weights
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = create_network(model_cfg)
model.load_state_dict(torch.load(model_path, map_location=device))

saved_path = "YOUR_SAVE_PATH"  # Path to save the results
run(model, video_dir, 
    channel_config={'channel_type': channel_type, 'snr': diff_snr}, 
    saved_path=saved_path, 
    target_size=(1080,1920), 
    max_frames=5, 
    max_batch=100,
    batch_size=1, 
    device=device)